# Notebook 9: K-Nearest Neighbours and Naive Bayes
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Explain the KNN algorithm and choose K
- Understand the curse of dimensionality
- Derive Bayes' theorem and understand conditional independence
- Apply Gaussian, Multinomial, and Bernoulli Naive Bayes
- Know when each algorithm is appropriate

## Part A: K-Nearest Neighbours (KNN)

## 1. The Algorithm

KNN is beautifully simple:
1. Compute the distance from the query point to every training point
2. Find the K closest points
3. **Classification**: take a majority vote
4. **Regression**: take the average

KNN makes **no assumptions about the data distribution** — it is a **non-parametric** method. The model *is* the training data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris, make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from matplotlib.colors import ListedColormap
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Simple KNN illustration
from sklearn.datasets import make_blobs
X_blob, y_blob = make_blobs(n_samples=100, centers=3, cluster_std=1.2, random_state=5)

knn3 = KNeighborsClassifier(n_neighbors=3).fit(X_blob, y_blob)

x_min, x_max = X_blob[:,0].min()-1, X_blob[:,0].max()+1
y_min, y_max = X_blob[:,1].min()-1, X_blob[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
Z = knn3.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cmap_light = ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF'])
cmap_bold  = ListedColormap(['#FF0000','#00AA00','#0000FF'])
ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
ax.scatter(X_blob[:,0], X_blob[:,1], c=y_blob, cmap=cmap_bold, edgecolors='white', s=50)

# Show query point and its 3 nearest neighbours
query = np.array([[1.0, 3.0]])
dists = np.sqrt(((X_blob - query)**2).sum(axis=1))
nearest_idx = np.argsort(dists)[:3]
for idx in nearest_idx:
    ax.plot([query[0,0], X_blob[idx,0]], [query[0,1], X_blob[idx,1]],
            'k--', alpha=0.5)
ax.scatter(*query[0], s=300, c='yellow', edgecolors='black', zorder=5, marker='*', label='Query')
ax.scatter(X_blob[nearest_idx,0], X_blob[nearest_idx,1],
           s=200, facecolors='none', edgecolors='black', linewidths=2, label='3 Nearest Neighbours')
ax.set_title('KNN: 3 Nearest Neighbours')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Choosing K — The Bias-Variance Trade-off

- **Small K** (e.g., K=1): very flexible, fits noise, high variance (overfitting)
- **Large K** (e.g., K=n): very smooth, underfitting, high bias

We use cross-validation to find the best K.

In [ ]:
X_m, y_m = make_moons(n_samples=400, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

K_values = range(1, 31)
train_acc, test_acc = [], []
for k in K_values:
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_tr, y_tr)
    train_acc.append(pipe.score(X_tr, y_tr))
    test_acc.append(pipe.score(X_te, y_te))

best_k = K_values[np.argmax(test_acc)]
print(f'Best K = {best_k}, Test Accuracy = {max(test_acc):.3f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_values, train_acc, label='Train accuracy', marker='o')
ax.plot(K_values, test_acc,  label='Test accuracy',  marker='s')
ax.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax.set_xlabel('K')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Accuracy vs K')
ax.legend()
plt.tight_layout()
plt.show()

## 3. The Curse of Dimensionality

KNN relies on **distance**. In high dimensions, all points become roughly equidistant — the notion of "nearest" breaks down.

> In 10 dimensions, you need exponentially more data to densely cover the space.

In [ ]:
# Demonstrate how fraction of volume in a hypersphere shrinks
dims = range(1, 30)
# Volume of unit hypersphere as a fraction of unit hypercube
import math
def vol_fraction(d, r=0.1):
    # Vol of hypersphere / Vol of hypercube [0,1]^d
    return (math.pi**(d/2) * r**d) / math.gamma(d/2 + 1)

fracs = [vol_fraction(d) for d in dims]
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(list(dims), fracs, marker='o')
ax.set_xlabel('Number of dimensions')
ax.set_ylabel('Volume fraction (log scale)')
ax.set_title('Curse of Dimensionality: Hypersphere Volume Fraction')
ax.set_title('Volume of unit hypersphere relative to hypercube shrinks exponentially')
plt.tight_layout()
plt.show()
print('At 20 dimensions, a hypersphere covers essentially 0% of the hypercube volume!')

## Part B: Naive Bayes

## 4. Bayes' Theorem

$$P(y | \mathbf{x}) = \frac{P(\mathbf{x} | y) \cdot P(y)}{P(\mathbf{x})}$$

- $P(y)$ — **prior**: how common is this class?
- $P(\mathbf{x} | y)$ — **likelihood**: how likely are these features given the class?
- $P(y | \mathbf{x})$ — **posterior**: what we want to predict

### The "Naive" Assumption
Features are **conditionally independent** given the class:

$$P(\mathbf{x} | y) = \prod_{j=1}^p P(x_j | y)$$

This is rarely true in practice, but Naive Bayes works surprisingly well despite it.

In [ ]:
# Manual Naive Bayes illustration with toy data
# Weather example: will someone play tennis?
print('Bayes Theorem Example: Will someone play tennis?\n')

# P(Sunny) = 5/14, P(Sunny|Play) = 2/9, P(Play) = 9/14
P_play  = 9/14
P_sunny = 5/14
P_sunny_given_play = 2/9

P_play_given_sunny = (P_sunny_given_play * P_play) / P_sunny
print(f'P(Play)             = {P_play:.3f}')
print(f'P(Sunny)            = {P_sunny:.3f}')
print(f'P(Sunny|Play)       = {P_sunny_given_play:.3f}')
print(f'P(Play|Sunny)       = {P_play_given_sunny:.3f}')
print(f'Decision: {"PLAY" if P_play_given_sunny > 0.5 else "NO PLAY"} (threshold 0.5)')

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report

# Gaussian NB for continuous features
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_tr, y_tr)
y_pred_gnb = gnb.predict(X_te)
print('Gaussian Naive Bayes on Breast Cancer:')
print(classification_report(y_te, y_pred_gnb, target_names=cancer.target_names))

In [ ]:
# Compare KNN vs Naive Bayes on the Iris dataset
from sklearn.datasets import load_iris
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_i, y_i, test_size=0.2, random_state=42)

models = [
    ('KNN (k=5)',       Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(5))])),
    ('KNN (k=15)',      Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(15))])),
    ('Gaussian NB',    GaussianNB()),
]

print(f'{'Model':<20} Train Acc   Test Acc   CV Acc (5-fold)')
print('-' * 60)
for name, model in models:
    model.fit(X_tr_i, y_tr_i)
    tr_acc = model.score(X_tr_i, y_tr_i)
    te_acc = model.score(X_te_i, y_te_i)
    cv_acc = cross_val_score(model, X_i, y_i, cv=5).mean()
    print(f'{name:<20} {tr_acc:.3f}       {te_acc:.3f}      {cv_acc:.3f}')

## 5. When to Use Each Algorithm

| | KNN | Naive Bayes |
|--|-----|--------------|
| **Speed (train)** | O(1) | O(n·p) |
| **Speed (predict)** | O(n·p) | O(K·p) |
| **Works well with** | Small, low-dim data | Text, large datasets |
| **Handles missing** | ❌ | ✅ (skip feature) |
| **Interpretable** | Limited | ✅ (priors/likelihoods) |
| **Key weakness** | Slow prediction, curse of dim | Strong independence assumption |

## Exercises

1. Implement **weighted KNN** where closer neighbours get more vote weight: `weight='distance'` in sklearn. Compare with uniform weights.
2. Use `KNeighborsRegressor` to predict California housing prices. What is the RMSE? Compare with linear regression.
3. The **class prior** in Naive Bayes can be manually adjusted. Experiment with `var_smoothing` in `GaussianNB` and observe its effect.
4. Build a **spam filter** using `MultinomialNB` on a small bag-of-words dataset (you'll revisit this in Notebook 12).

## Reflection Questions
- KNN has no explicit training step — is that an advantage or a disadvantage in production?
- Why does Naive Bayes often work well on text classification despite the obviously violated independence assumption?
- What is Laplace smoothing in Naive Bayes and why is it necessary?

---
**Next →** `10_decision_trees.ipynb`